# Week 54, Day 1: LCS and edit distance

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w054d1.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
a, b = "ABCBDAB", "BDCABA"
L = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
for i in range(1, len(a) + 1):
    for j in range(1, len(b) + 1):
        if a[i - 1] == b[j - 1]:
            L[i][j] = L[i - 1][j - 1] + 1
        else:
            L[i][j] = max(L[i - 1][j], L[i][j - 1])
print(L[-1][-1])
print(L[3])

In [ ]:
def lcs(a, b):
    n, m = len(a), len(b)
    L = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if a[i - 1] == b[j - 1]:
                L[i][j] = L[i - 1][j - 1] + 1
            else:
                L[i][j] = max(L[i - 1][j], L[i][j - 1])
    out, i, j = [], n, m
    while i and j:
        if a[i - 1] == b[j - 1]:
            out.append(a[i - 1])
            i, j = i - 1, j - 1
        elif L[i - 1][j] >= L[i][j - 1]:
            i -= 1
        else:
            j -= 1
    return "".join(reversed(out))

print(lcs("ABCBDAB", "BDCABA"), lcs("AGGTAB", "GXTXAYB"))

In [ ]:
def diff(old, new):
    n, m = len(old), len(new)
    L = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            L[i][j] = L[i - 1][j - 1] + 1 if old[i - 1] == new[j - 1] else max(L[i - 1][j], L[i][j - 1])
    out, i, j = [], n, m
    while i or j:
        if i and j and old[i - 1] == new[j - 1]:
            out.append("  " + old[i - 1])
            i, j = i - 1, j - 1
        elif j and (not i or L[i][j - 1] >= L[i - 1][j]):
            out.append("+ " + new[j - 1])
            j -= 1
        else:
            out.append("- " + old[i - 1])
            i -= 1
    return out[::-1]

old = ["eggs", "milk", "bread", "tea"]
new = ["eggs", "bread", "butter", "tea", "jam"]
print("\n".join(diff(old, new)))

In [ ]:
def edits(a, b):
    n, m = len(a), len(b)
    D = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1):
        D[i][0] = i
    for j in range(m + 1):
        D[0][j] = j
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            same = a[i - 1] == b[j - 1]
            D[i][j] = min(D[i - 1][j - 1] + (0 if same else 1), D[i - 1][j] + 1, D[i][j - 1] + 1)
    steps, i, j = [], n, m
    while i or j:
        if i and j and D[i][j] == D[i - 1][j - 1] + (a[i - 1] != b[j - 1]):
            if a[i - 1] != b[j - 1]:
                steps.append(f"replace {a[i - 1]} with {b[j - 1]}")
            i, j = i - 1, j - 1
        elif i and D[i][j] == D[i - 1][j] + 1:
            steps.append(f"delete {a[i - 1]}")
            i -= 1
        else:
            steps.append(f"insert {b[j - 1]}")
            j -= 1
    return D[n][m], steps[::-1]

print(edits("kitten", "sitting"))

> **Key idea.** Tabulate string DPs on prefixes, with an extra row and column for the empty prefix. The LCS table adds 1 diagonally on a match and takes the larger of up and left otherwise; edit distance starts its edges at i and j and takes the cheapest of a diagonal match or replacement, a deletion (up) or an insertion (left). Walk back from the bottom-right corner to recover the subsequence, a line-by-line diff, or the list of edits, choosing at each cell an option that produced its value. Supersequences, palindromic subsequences and deletion counts are these tables in disguise. Two rows suffice when you only need the number.

### Exercise: A common thread

Write `lcs(a, b)`: one longest common subsequence of two sequences (strings or lists), returned as the same type as `a` (a string for strings, a list for lists). Walk back preferring a diagonal match, then **up**, then left. Then `lcs_length(a, b)`: the length only, with **two rows**.

In [ ]:
def lcs(a, b):
    ___

def lcs_length(a, b):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import itertools, random\ndef is_subseq(x, y):\n    it = iter(y)\n    return all(ch in it for ch in x)\ndef test_lcs():\n    \"lcs\"\n    assert lcs(\"ABCBDAB\", \"BDCABA\") == \"BCBA\" and lcs(\"AGGTAB\", \"GXTXAYB\") == \"GTAB\"\n    assert lcs(\"\", \"abc\") == \"\" and lcs([1, 2, 3], [2, 3, 4]) == [2, 3]\n    rng = random.Random(1)\n    for _ in range(150):\n        a = \"\".join(rng.choice(\"abc\") for _ in range(rng.randint(0, 8)))\n        b = \"\".join(rng.choice(\"abc\") for _ in range(rng.randint(0, 8)))\n        best = max((r for r in range(len(a) + 1) for c in itertools.combinations(a, r) if is_subseq(c, b)), default=0)\n        s = lcs(a, b)\n        assert len(s) == best == lcs_length(a, b) and is_subseq(s, a) and is_subseq(s, b)\n    a = \"\".join(rng.choice(\"ACGT\") for _ in range(600))\n    b = \"\".join(rng.choice(\"ACGT\") for _ in range(600))\n    assert lcs_length(a, b) == len(lcs(a, b))")

### Exercise: What changed?

Write `diff(old, new)` for two lists of lines: a list of strings, each a line prefixed by `"  "` (kept), `"- "` (removed) or `"+ "` (added), from an LCS table walked back. At each step back: a match goes diagonally; otherwise output an **addition** if moving left keeps at least as much LCS as moving up, else a removal (so, reading forwards, removals come before additions in a changed block). Then `patch(old, changes)`: apply a diff to the old lines, returning the new lines (raise `ValueError` if a kept or removed line doesn't match).

In [ ]:
def diff(old, new):
    ___

def patch(old, changes):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_diff():\n    \"diffs\"\n    old = [\"eggs\", \"milk\", \"bread\", \"tea\"]\n    new = [\"eggs\", \"bread\", \"butter\", \"tea\", \"jam\"]\n    d = diff(old, new)\n    assert d == [\"  eggs\", \"- milk\", \"  bread\", \"+ butter\", \"  tea\", \"+ jam\"], d\n    assert patch(old, d) == new\n    assert diff([], [\"a\"]) == [\"+ a\"] and diff([\"a\"], []) == [\"- a\"] and diff([\"x\"], [\"x\"]) == [\"  x\"]\n    assert diff([\"a\", \"b\"], [\"c\"]) == [\"- a\", \"- b\", \"+ c\"]\n    try:\n        patch([\"x\"], [\"- y\"])\n        assert False\n    except ValueError:\n        pass\n    rng = random.Random(2)\n    for _ in range(100):\n        old = [rng.choice(\"abcde\") for _ in range(rng.randint(0, 8))]\n        new = [rng.choice(\"abcde\") for _ in range(rng.randint(0, 8))]\n        d = diff(old, new)\n        assert patch(old, d) == new\n        kept = sum(1 for line in d if line.startswith(\"  \"))\n        n, m = len(old), len(new)\n        L = [[0] * (m + 1) for _ in range(n + 1)]\n        for i in range(1, n + 1):\n            for j in range(1, m + 1):\n                L[i][j] = L[i - 1][j - 1] + 1 if old[i - 1] == new[j - 1] else max(L[i - 1][j], L[i][j - 1])\n        assert kept == L[n][m]")

### Exercise: Edits with prices

Write `edit_cost(a, b, insert=1, delete=1, replace=1)`: the cheapest way to turn a into b with those operation costs (keeping an equal character is free). Then `edit_steps(a, b)`: with unit costs, a list of the edits as tuples `("insert", position, char)`, `("delete", position, char)` or `("replace", position, old, new)`, where positions refer to the string **as it is when that edit is applied**, applying them in order from left to right. The checks apply your steps to a and expect b.

In [ ]:
def edit_cost(a, b, insert=1, delete=1, replace=1):
    ___

def edit_steps(a, b):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef apply(a, steps):\n    s = list(a)\n    for step in steps:\n        if step[0] == \"insert\":\n            s.insert(step[1], step[2])\n        elif step[0] == \"delete\":\n            assert s[step[1]] == step[2]\n            del s[step[1]]\n        else:\n            assert s[step[1]] == step[2]\n            s[step[1]] = step[3]\n    return \"\".join(s)\ndef ref(a, b, ins, dele, rep):\n    n, m = len(a), len(b)\n    D = [[0] * (m + 1) for _ in range(n + 1)]\n    for i in range(1, n + 1):\n        D[i][0] = i * dele\n    for j in range(1, m + 1):\n        D[0][j] = j * ins\n    for i in range(1, n + 1):\n        for j in range(1, m + 1):\n            D[i][j] = min(D[i - 1][j - 1] + (0 if a[i - 1] == b[j - 1] else rep), D[i - 1][j] + dele, D[i][j - 1] + ins)\n    return D[n][m]\ndef test_edits():\n    \"edits\"\n    assert edit_cost(\"kitten\", \"sitting\") == 3 and edit_cost(\"\", \"abc\") == 3 and edit_cost(\"abc\", \"\") == 3\n    assert edit_cost(\"abc\", \"abd\", replace=5) == 2 and edit_cost(\"ab\", \"ba\", insert=2, delete=2, replace=1) == 2\n    steps = edit_steps(\"kitten\", \"sitting\")\n    assert len(steps) == 3 and apply(\"kitten\", steps) == \"sitting\"\n    rng = random.Random(3)\n    for _ in range(150):\n        a = \"\".join(rng.choice(\"abc\") for _ in range(rng.randint(0, 7)))\n        b = \"\".join(rng.choice(\"abc\") for _ in range(rng.randint(0, 7)))\n        costs = (rng.randint(1, 3), rng.randint(1, 3), rng.randint(1, 4))\n        assert edit_cost(a, b, *costs) == ref(a, b, *costs)\n        steps = edit_steps(a, b)\n        assert len(steps) == ref(a, b, 1, 1, 1) and apply(a, steps) == b, (a, b, steps)")

### Exercise: Hidden palindromes

Write `longest_palindrome_subseq(s)`: one longest palindromic subsequence of s (a palindrome you get by deleting characters). Its length is the LCS of s and its reverse, but an LCS of the two isn't always a palindrome itself, so build it with an **interval** DP instead: P(i, j) is the length for `s[i:j + 1]`; if the end characters match they wrap the best of the inside, otherwise drop one end. Fill i downwards and j upwards, then walk inwards from (0, n − 1), collecting matched ends. Then `fewest_insertions(s)`: the fewest characters to insert anywhere to make s a palindrome (len(s) minus that length).

In [ ]:
def longest_palindrome_subseq(s):
    ___

def fewest_insertions(s):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import itertools, random\ndef test_pal():\n    \"palindromes\"\n    assert len(longest_palindrome_subseq(\"bbbab\")) == 4 and longest_palindrome_subseq(\"cbbd\") == \"bb\"\n    assert longest_palindrome_subseq(\"\") == \"\" and longest_palindrome_subseq(\"a\") == \"a\"\n    assert fewest_insertions(\"mbadm\") == 2 and fewest_insertions(\"leetcode\") == 5 and fewest_insertions(\"aba\") == 0\n    rng = random.Random(4)\n    for _ in range(100):\n        s = \"\".join(rng.choice(\"abc\") for _ in range(rng.randint(0, 9)))\n        best = max((r for r in range(len(s) + 1) for c in itertools.combinations(s, r) if c == c[::-1]), default=0)\n        p = longest_palindrome_subseq(s)\n        it = iter(s)\n        assert len(p) == best and p == p[::-1] and all(ch in it for ch in p), (s, p)\n        assert fewest_insertions(s) == len(s) - best")

### Exercise: One string containing both

Write `shortest_supersequence(a, b)`: one shortest string that has both a and b as subsequences, by walking back through the LCS table: a match contributes its character once; otherwise take the character of the side you step away from (prefer stepping **up**, which takes `a[i - 1]`); when one string runs out, take the rest of the other.

In [ ]:
def shortest_supersequence(a, b):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef is_subseq(x, y):\n    it = iter(y)\n    return all(ch in it for ch in x)\ndef test_super():\n    \"supersequences\"\n    s = shortest_supersequence(\"abac\", \"cab\")\n    assert len(s) == 5 and is_subseq(\"abac\", s) and is_subseq(\"cab\", s)\n    assert shortest_supersequence(\"\", \"xy\") == \"xy\" and shortest_supersequence(\"abc\", \"abc\") == \"abc\"\n    rng = random.Random(5)\n    for _ in range(150):\n        a = \"\".join(rng.choice(\"abc\") for _ in range(rng.randint(0, 8)))\n        b = \"\".join(rng.choice(\"abc\") for _ in range(rng.randint(0, 8)))\n        n, m = len(a), len(b)\n        L = [[0] * (m + 1) for _ in range(n + 1)]\n        for i in range(1, n + 1):\n            for j in range(1, m + 1):\n                L[i][j] = L[i - 1][j - 1] + 1 if a[i - 1] == b[j - 1] else max(L[i - 1][j], L[i][j - 1])\n        s = shortest_supersequence(a, b)\n        assert len(s) == n + m - L[n][m] and is_subseq(a, s) and is_subseq(b, s), (a, b, s)")

### Bug hunt: One character out

Row i of the table adds the character `a[i - 1]`, but this compares `a[i]` and `b[j]`, so it checks the wrong characters (and would run off the end if the loops reached the last row). It prints `3` instead of `4` for `"ABCBDAB"` and `"BDCABA"`.

In [ ]:
a, b = "ABCBDAB", "BDCABA"
L = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
for i in range(1, len(a)):
    for j in range(1, len(b)):
        if a[i] == b[j]:
            L[i][j] = L[i - 1][j - 1] + 1
        else:
            L[i][j] = max(L[i - 1][j], L[i][j - 1])
print(max(max(row) for row in L))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"prefix indexing\"\n    expect_output(\"4\")")

### Bug hunt: Free deletions

The edit-distance table's edges should be D(i, 0) = i and D(0, j) = j: turning a prefix into nothing costs a deletion per character. Leaving them at 0 makes deleting or inserting a whole prefix free, so `"abc"` to `""` costs nothing and this prints `[0, 3]`. Fill the edges so it prints `[3, 3]`.

In [ ]:
def edit_distance(a, b):
    n, m = len(a), len(b)
    D = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            same = a[i - 1] == b[j - 1]
            D[i][j] = min(D[i - 1][j - 1] + (0 if same else 1), D[i - 1][j] + 1, D[i][j - 1] + 1)
    return D[n][m]

print([edit_distance("abc", ""), edit_distance("kitten", "sitting")])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"the edges\"\n    expect_output(\"[3, 3]\")")

### Bug hunt: Two rows, one forgotten edge

This two-row edit distance starts each new row with 0, but the first entry of row i is D(i, 0) = i. So deleting the leading `"ab"` costs nothing, and it prints `0` for `"abc"` to `"c"`. Start each row with i so it prints `2`.

In [ ]:
def edit_distance(a, b):
    prev = list(range(len(b) + 1))
    for i in range(1, len(a) + 1):
        cur = [0] * (len(b) + 1)
        for j in range(1, len(b) + 1):
            same = a[i - 1] == b[j - 1]
            cur[j] = min(prev[j - 1] + (0 if same else 1), prev[j] + 1, cur[j - 1] + 1)
        prev = cur
    return prev[-1]

print(edit_distance("abc", "c"))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"each row's edge\"\n    expect_output(\"2\")")

### Puzzle: The LCS walk back

Arrange the lines that walk back through a filled LCS table `L`. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the LCS length of "abcd" and "acbd"
q = ___   # the edit distance from "sunday" to "saturday"
r = ___   # D(3, 0) in any edit-distance table
s = ___   # the length of the shortest common supersequence of "abc" and "bcd"
t = ___   # the length of the longest palindromic subsequence of "character"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 3 and q == 3 and r == 3, \"p, q and r\"\n    assert s == 4 and t == 5, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: did you mean?

Typing errors are usually one of four kinds: a missing letter, an extra letter, a wrong letter, or two neighbouring letters **swapped** ("teh" for "the"). Levenshtein distance counts a swap as two edits; Fred Damerau (1964) counted it as one. Write:

- `typo_distance(a, b)`: the edit distance where a swap of two **adjacent** characters also costs 1 (the "optimal string alignment" version: add to the recurrence D(i − 2, j − 2) + 1 when a[i − 1] == b[j − 2] and a[i − 2] == b[j − 1]).
- `suggest(word, dictionary, limit=2, count=3)`: up to `count` dictionary words within `typo_distance` of `limit`, nearest first, ties alphabetical. The word itself, if in the dictionary, comes first (distance 0).
- `suggest_fast(word, dictionary, limit=2, count=3)`: the same answers, faster: skip any dictionary word whose length differs from the word's by more than `limit` (it can't be close enough), and stop filling a word's table as soon as a whole row exceeds `limit` (no later entry can come back under it).

In [ ]:
def typo_distance(a, b):
    ___

def suggest(word, dictionary, limit=2, count=3):
    ___

def suggest_fast(word, dictionary, limit=2, count=3):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random, time\ndef test_typos():\n    \"swaps\"\n    assert typo_distance(\"teh\", \"the\") == 1 and typo_distance(\"ca\", \"abc\") == 3\n    assert typo_distance(\"kitten\", \"sitting\") == 3 and typo_distance(\"\", \"ab\") == 2 and typo_distance(\"ab\", \"ba\") == 1\n    assert typo_distance(\"abcdef\", \"badcfe\") == 3\n    words = [\"the\", \"then\", \"they\", \"tea\", \"ten\", \"hen\", \"theme\"]\n    assert suggest(\"teh\", words) == [\"tea\", \"ten\", \"the\"]\n    assert suggest(\"the\", words, count=2) == [\"the\", \"then\"]\n    assert suggest(\"xyzzy\", words) == []\n    rng = random.Random(6)\n    letters = \"abcde\"\n    dictionary = sorted({\"\".join(rng.choice(letters) for _ in range(rng.randint(2, 7))) for _ in range(400)})\n    for _ in range(40):\n        w = \"\".join(rng.choice(letters) for _ in range(rng.randint(2, 7)))\n        assert suggest_fast(w, dictionary) == suggest(w, dictionary)\n        assert suggest_fast(w, dictionary, limit=1, count=5) == suggest(w, dictionary, limit=1, count=5)\ndef test_fast():\n    \"the fast version prunes\"\n    rng = random.Random(7)\n    dictionary = [\"\".join(rng.choice(\"abcdefghij\") for _ in range(rng.randint(3, 14))) for _ in range(3000)]\n    t0 = time.perf_counter()\n    for w in (\"abcdefg\", \"hijabc\", \"jjjjjjjj\"):\n        suggest(w, dictionary)\n    slow = time.perf_counter() - t0\n    t0 = time.perf_counter()\n    for w in (\"abcdefg\", \"hijabc\", \"jjjjjjjj\"):\n        suggest_fast(w, dictionary)\n    fast = time.perf_counter() - t0\n    assert fast < slow / 2, (fast, slow)")

## Warm-up

`L[i][j]` is the length of the LCS of the first i characters of `a` and the first j of `b`. Row 3 is about `"ABC"` against every prefix of `"BDCABA"`.

## The LCS table

Week 16's recursion worked on **suffixes** (positions i and j onwards). Tabulating is cleaner on **prefixes**:

**L(i, j) = the length of the LCS of a[:i] and b[:j]**,

with L(0, j) = L(i, 0) = 0 (an empty string has nothing in common with anything). For i, j ≥ 1:

- if the last characters match, `a[i - 1] == b[j - 1]`, they can end the common subsequence: L(i, j) = L(i − 1, j − 1) + 1;
- otherwise at least one of them isn't used: L(i, j) = max(L(i − 1, j), L(i, j − 1)).

Each entry reads the one above, the one to the left and the diagonal, so row-major order works, exactly as in Thursday's grids. Time and memory O(n × m). For the length alone, two rows suffice.

Notice the indexing: row i is about the first i characters, so the character it adds is `a[i - 1]`. The extra row and column of zeros for empty prefixes are what make the recurrence work without special cases.

## Reading it back

To recover a longest common subsequence, start at (n, m) and walk back:

- if `a[i - 1] == b[j - 1]`, that character is in the LCS: record it and step diagonally to (i − 1, j − 1);
- otherwise step to whichever of (i − 1, j) and (i, j − 1) has the larger value (on a tie, either works; choose a fixed rule, such as up first, for predictable answers).

Stop when i or j reaches 0, and reverse what you recorded. There may be many different LCSs of the same length; the tie rule picks one.

## Diff

A **diff** shows how one text became another: lines kept, lines removed (`-`) and lines added (`+`). Treat each **line** as one symbol and find the LCS of the two lists of lines: those are the lines kept, and everything else was removed from the old text or added in the new one. Walking back through the table produces the diff from the end, so collect it and reverse.

The classic Unix `diff` (Douglas McIlroy and James Hunt, 1976) was built on this idea; Git's default is Eugene Myers' 1986 algorithm, which finds the same kind of answer faster when the texts are similar.

## Edit distance, with the edits

The **edit distance** (Vladimir Levenshtein, 1965) counts the fewest insertions, deletions and replacements turning a into b. On prefixes:

**D(i, j) = the edit distance from a[:i] to b[:j]**,

with D(i, 0) = i (delete everything) and D(0, j) = j (insert everything). For i, j ≥ 1, the last characters are matched (free if equal, a replacement if not), or a[i − 1] is deleted, or b[j − 1] is inserted:

D(i, j) = min(D(i − 1, j − 1) + (0 if equal else 1), D(i − 1, j) + 1, D(i, j − 1) + 1).

Different **costs** per operation fit in directly: replace the 1s with the costs. Spell checkers often make a replacement between neighbouring keys cheaper.

To list the edits, walk back from (n, m) choosing, at each cell, an option that produced its value: diagonal (keep or replace), up (delete) or left (insert).

## In disguise

Many string problems are one of these two tables:

- **Shortest common supersequence** (the shortest string containing both as subsequences): its length is n + m − LCS, and walking back through the LCS table builds it.
- **Longest palindromic subsequence** of s: the LCS of s and its reverse.
- **Fewest deletions** to make two strings equal: n + m − 2 × LCS.
- **Is a a subsequence of b?** LCS equals len(a), though two pointers do it in O(n + m).
- **Fuzzy matching**, "did you mean": the dictionary words with the smallest edit distance.

When only the number is needed and the strings are long, keep two rows: O(m) memory instead of O(n × m).

## Practice

## Quiz

## Daily challenge

## Before you go

Write two short versions of a sentence, then work out (by hand, or with your `diff`) which words were kept, removed and added. Tomorrow: the longest increasing subsequence, and a beautiful O(n log n) method based on a game of patience.